# AWS Module 6 – Storage
**Status:** ⬜ Not started | **Course:** AWS Cloud Practitioner Essentials

**Lessons:** Intro to Storage → Instance Store & EBS → EBS Data Lifecycle → Amazon S3 → S3 Storage Classes & Lifecycle → S3 Demo → Amazon EFS → Amazon FSx (reading) → Storage Gateway → Elastic Disaster Recovery (reading) → Comparing Storage Services

---

## MODULE 6 INTRODUCTION

### Introduction to Storage
AWS storage replaces buying and managing disks in your own data center, and it **scales with your needs**. There are three types:

| Type | How data is stored | Analogy | AWS services |
|---|---|---|---|
| **Block** | Data split into fixed-size **blocks**; only changed blocks are rewritten | A hard drive plugged into your computer | Instance store, **Amazon EBS** |
| **File** | Data in **files and folders** (hierarchy), shared by many users/servers | A shared network drive | **Amazon EFS**, **Amazon FSx** |
| **Object** | Each **object** = data + metadata + unique key, stored in a flat structure | A huge valet coat check: hand over the item, get a ticket (key) back | **Amazon S3** |

---

## STORAGE

### EC2 Instance Store and Amazon Elastic Block Store (Amazon EBS)

| | **Instance store** | **Amazon EBS** |
|---|---|---|
| What | Disk **physically attached** to the host running your EC2 instance | **Network-attached** virtual hard drive (a "volume") for EC2 |
| Persistence | **Temporary**: data is **lost** when the instance stops or terminates | **Persistent**: data survives stop/start |
| Location | Tied to that physical host | Lives in **one AZ**; must be in the same AZ as its instance |
| Best for | Temporary data: caches, buffers, scratch files | Databases, boot volumes, anything you need to keep |

**EBS details:**

- You choose size, type, and performance; you can **resize** or change type on the fly, and detach/attach to another instance
- **Volume types:** **SSD-backed** (general purpose `gp3`, provisioned IOPS `io2` for demanding databases) and **HDD-backed** (throughput `st1`, cold `sc1` for cheaper, infrequent access)

### Amazon Elastic Block Store (Amazon EBS) Data Lifecycle

- **EBS snapshot** = a point-in-time **backup** of a volume
  - **Incremental:** the first snapshot copies everything; later ones save only the **blocks that changed**
  - Stored redundantly (in Amazon S3, across multiple AZs)
  - Use a snapshot to **create a new volume in another AZ**, or **copy it to another Region** for disaster recovery
- **Amazon Data Lifecycle Manager** = automates snapshot **creation, retention, and deletion** on a schedule (e.g., nightly backups, keep 30 days) to control cost
- **AWS Backup** = centrally manages and automates backups across many AWS services (EBS, RDS, DynamoDB, EFS, and more)

### Amazon Simple Storage Service (Amazon S3)

- **Object storage** for virtually **unlimited** data; a single object can be up to **5 TB**
- Objects are stored in **buckets** (bucket names are globally unique; each bucket lives in a Region you choose)
- Designed for **99.999999999% (11 nines) durability**: data is stored across multiple AZs
- Key features:
  - **Versioning:** keep previous versions; recover from accidental deletes/overwrites
  - **Access control:** bucket policies and IAM; **Block Public Access** is on by default
  - **Encryption:** new objects are encrypted by default
  - **Replication:** copy objects to another bucket/Region automatically
  - **Static website hosting**
- Common uses: backups, data lakes, media, logs, static websites

### Amazon S3 Storage Classes and S3 Lifecycle
Trade-off: **cheaper storage = slower or costlier retrieval**.

| Storage class | Use when… | Notes |
|---|---|---|
| **S3 Standard** | Data accessed **frequently** | Default; multi-AZ; millisecond access |
| **S3 Intelligent-Tiering** | Access patterns are **unknown or changing** | Automatically moves objects between tiers; small monitoring fee |
| **S3 Express One Zone** | You need the **fastest** performance | Single-digit-millisecond access; single AZ |
| **S3 Standard-IA** | **Infrequent** access but needs fast retrieval | Lower storage cost, retrieval fee |
| **S3 One Zone-IA** | Infrequent, **re-creatable** data | Single AZ, cheaper than Standard-IA |
| **S3 Glacier Instant Retrieval** | Archive accessed about once a quarter, needs milliseconds | Archive pricing, instant access |
| **S3 Glacier Flexible Retrieval** | Archive where minutes to hours is fine | Low cost |
| **S3 Glacier Deep Archive** | Archive accessed **once or twice a year** (e.g., 7–10 yr compliance) | **Lowest cost**; retrieval within hours |

> 💡 Infrequent-access and Glacier classes charge **retrieval fees** and have **minimum storage durations**, so don't use them for data you read often.

**S3 Lifecycle policies** automate moving data between classes:

- **Transition actions:** e.g., Standard → Standard-IA after 30 days → Glacier Flexible Retrieval after 90 days
- **Expiration actions:** e.g., delete after 7 years

### Amazon S3 Demonstration
Typical steps in the console:

1. **Create a bucket** (unique name + Region; keep Block Public Access on)
2. **Upload objects** (files) and view their properties/metadata
3. Set **permissions** (bucket policy) as needed
4. Turn on **versioning**
5. Add a **lifecycle rule** to transition or expire objects

### Amazon Elastic File System (Amazon EFS)

- **Managed, elastic file system** (NFS) for **Linux** workloads
- **Many EC2 instances** (and containers, Lambda, on-prem servers) can read and write the **same files at the same time**
- **Scales automatically** as you add or remove files; pay only for what you store
- **Regional** (data stored across multiple AZs) or **One Zone** options
- Storage classes + lifecycle management move cold files to cheaper tiers

### Amazon FSx (reading)
Fully managed file systems built on popular commercial and open-source file systems:

| Option | Use it for |
|---|---|
| **FSx for Windows File Server** | Windows apps needing SMB file shares and Active Directory integration |
| **FSx for Lustre** | High-performance computing (HPC), ML training; integrates with S3 |
| **FSx for NetApp ONTAP** | Moving NetApp workloads to AWS with familiar features |
| **FSx for OpenZFS** | Moving ZFS/Linux file servers to AWS |

### AWS Storage Gateway
**Hybrid** storage: connects on-premises apps to cloud storage, so local systems can use AWS storage with low-latency local caching.

| Gateway type | What it does |
|---|---|
| **S3 File Gateway** | On-prem apps save files (NFS/SMB) that are stored as objects in S3 |
| **Volume Gateway** | Block storage volumes for on-prem apps, backed up to AWS |
| **Tape Gateway** | Replaces physical backup tapes with virtual tapes in S3/Glacier |

### AWS Elastic Disaster Recovery (reading)

- Continuously **replicates servers** (on-prem or in the cloud) into a low-cost **staging area** in AWS
- If disaster strikes, **launch recovered servers in AWS within minutes**
- Two DR terms to know:
  - **RPO (Recovery Point Objective)** = how much data you can afford to lose (measured in time). Elastic DR achieves RPO in **seconds**
  - **RTO (Recovery Time Objective)** = how fast you must be back up. Elastic DR achieves RTO in **minutes**

### Comparing Storage Services

| Need | Choose |
|---|---|
| Temporary scratch data on one instance | **Instance store** |
| Persistent disk for one EC2 instance (database, OS, apps that change small parts of files) | **Amazon EBS** |
| Shared files for many Linux instances | **Amazon EFS** |
| Windows file shares, HPC, NetApp, or ZFS | **Amazon FSx** |
| Massive amounts of data, write once / read many, web access, data lakes | **Amazon S3** |
| On-prem apps using cloud storage | **AWS Storage Gateway** |

**Classic exam comparison, EBS vs S3:**

- **EBS:** good when you make **frequent small edits** to a file (only changed blocks are written); limited to one AZ
- **S3:** good for **whole objects** you write once and read many times (photos, videos, backups); any change re-uploads the whole object; virtually unlimited and highly durable

---

## MODULE 6 CONCLUSION

### Module Summary

- **Block** = instance store (temporary) + **EBS** (persistent, single AZ, snapshots)
- **Object** = **S3** (buckets, 11 nines durability, storage classes, lifecycle policies)
- **File** = **EFS** (Linux, shared, elastic) + **FSx** (Windows, Lustre, NetApp, OpenZFS)
- **Hybrid** = Storage Gateway | **DR** = Elastic Disaster Recovery, Backup, snapshots

---

## 🎯 Databricks SA Connection

- **S3 is the foundation of the lakehouse on AWS.** Databricks stores tables as open-format **Delta Lake / Apache Iceberg** files in the **customer's own S3 buckets**. The customer keeps ownership of their data, with no lock-in
- **Separation of storage and compute:** cheap, durable storage (S3) + compute you pay for only while it runs. This is a core selling point
- **Unity Catalog** governs S3 data through **storage credentials** (IAM roles), **external locations**, and **volumes** (for non-tabular files like PDFs and images)
- **Storage classes in cost conversations:** move old raw data to cheaper tiers with lifecycle rules, but **don't archive files that active tables still read** (queries would fail or get slow and expensive)
- **EBS and instance store on classic clusters:** cluster nodes use EBS volumes, and storage-optimized instances with local SSDs speed up Databricks' **disk cache**
- **Disaster recovery:** RPO/RTO are the same terms customers use when designing Databricks DR across Regions
- **Discovery question:** "Where does your data live today, and in what format?" (S3? HDFS on-prem? A data warehouse? Parquet, CSV, JSON?)

---

## 📝 Exam / Interview Quick Hits

- Block = **EBS** | Object = **S3** | File = **EFS / FSx**
- Instance store = **temporary** | EBS = **persistent, single AZ**; snapshots are **incremental**
- S3: buckets + objects, max object **5 TB**, **11 nines** durability
- Unknown access → **Intelligent-Tiering** | Cheapest archive → **Glacier Deep Archive**
- Lifecycle policies = automatic **transition** + **expiration**
- **EFS** = shared Linux files | **FSx for Windows** = SMB/Active Directory | **FSx for Lustre** = HPC
- Hybrid = **Storage Gateway** | RPO = data loss tolerance, RTO = downtime tolerance